# SetFit

In [ ]:
# pip install setfit sentence_transformers

In [ ]:
# This version works well (so far)
# pip install "transformers>=4.20.0,<5.0.0" "setfit==1.1.3" -q

# You may have to restart session after installing

In [1]:
from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments  # ← new API
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

c:\Users\matthieu.catteyfaye\Documents\NLP\.venv_nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load and open the dataset from this path '/content/tweets_dataset.csv' and split the dataset into training and evaluation. the training set is randomly selected from 8 text in the dataset.
df = pd.read_csv('../data/tweets_dataset.csv', on_bad_lines='skip')
df.head()

,name,tweet,is_real
0,Billie Eilish,i’m fine. just floating in a hoodie-shaped clo...,False
1,Ryan Reynolds,I watched Frozen without my two-year-old this ...,True
2,Billie Eilish,people really be like “you’ve changed” like th...,False
3,Billie Eilish,people really be like “you’ve changed” like th...,False
4,Elon Musk,Nuke Mars!,True


In [3]:
# Rename columns
df = df.rename(columns={'tweet': 'text', 'is_real': 'label'})

# Map True/False in 'label' column to 0/1
df['label'] = df['label'].map({True: 1, False: 0})

df.head()

,name,text,label
0,Billie Eilish,i’m fine. just floating in a hoodie-shaped clo...,0.0
1,Ryan Reynolds,I watched Frozen without my two-year-old this ...,1.0
2,Billie Eilish,people really be like “you’ve changed” like th...,0.0
3,Billie Eilish,people really be like “you’ve changed” like th...,0.0
4,Elon Musk,Nuke Mars!,1.0


In [4]:
print(df['label'].value_counts(normalize=False))

label
1.0    31
0.0    28
Name: count, dtype: int64


In [5]:
# To ensure a balanced proportion of labels, perform stratified sampling for the training set.
# We want 8 texts for training, so we'll aim for 4 of each label if possible.

# Sample 4 texts where label is 0
train_df_label_0 = df[df['label'] == 0].sample(n=4, random_state=42)

# Sample 4 texts where label is 1
train_df_label_1 = df[df['label'] == 1].sample(n=4, random_state=42)

# Concatenate to form the balanced training DataFrame
train_df = pd.concat([train_df_label_0, train_df_label_1])

# Shuffle the training DataFrame to mix the labels
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)

# The remaining data (for test and eval)
remaining_df = df.drop(train_df.index)

# Split remaining into test and eval (50/50 split)
test_df = remaining_df.sample(frac=0.5, random_state=42)
eval_df = remaining_df.drop(test_df.index)

# Convert to Hugging Face datasets
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))
eval_dataset = Dataset.from_pandas(eval_df.reset_index(drop=True))

# Check sizes
print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}, Eval: {len(eval_dataset)}")
print("Train label distribution:")
print(train_df['label'].value_counts(normalize=False))

Train: 8, Test: 49, Eval: 49
Train label distribution:
label
0.0    4
1.0    4
Name: count, dtype: int64


In [6]:
# Prepare the evaluation metrics

def compute_metrics(y_pred, y_test):
    accuracy = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="weighted"
    )
    return {
        "accuracy":  round(accuracy, 4),
        "precision": round(precision, 4),
        "recall":    round(recall, 4),
        "f1":        round(f1, 4),
    }

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true" # If you don't want to use W&B

output_dir = "/content/setfit_2epochs" # Create a folder to store the fined-tuned model

# Load SetFit model from Hub
model = SetFitModel.from_pretrained(
    "sentence-transformers/paraphrase-mpnet-base-v2")

# Setting
args = TrainingArguments(
    batch_size=16,
    num_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    metric=compute_metrics,
)

# Train and evaluate
trainer.train()
metrics = trainer.evaluate()
print(">>>> Evaluation Result:", metrics)

# Save the model
trainer.model.save_pretrained(output_dir)

# **Understanding the Training Arguments:**

`batch_size=16` — The model learns from 16 examples at a time, rather than all at once.

`num_epochs=2` — It reads through the entire training data 2 times to reinforce learning.

`eval_strategy="epoch"` — After each full read-through, test the model on unseen examples to check its progress.

`save_strategy="epoch"` — Take a snapshot of the model after each test.

`load_best_model_at_end=True` — When training finishes, roll back to whichever snapshot performed best — not necessarily the last one.

---

# **Understanding the values produced during the Training process:**
`Epoch` — Which round you're on out of 4 total.

`Training Loss` — How many mistakes the model makes on the data it's learning from. Lower is better. This should drop steadily each epoch.

`Validation Loss` — How many mistakes it makes on data it's never seen before. This is the more honest score — it tells you if the model truly learned or just memorized.

---

### What to watch for

**✅ Healthy training** — Both losses drop together over epochs.

**⚠️ Overfitting** — Training loss keeps dropping but Validation loss starts *rising* (like epochs 3→4 above). The model is memorizing the training data rather than learning general patterns. This is exactly why `load_best_model_at_end=True` exists.

**⚠️ Underfitting** — Both losses are still high even at the last epoch. The model hasn't learned enough — try more epochs or a larger model.

In [ ]:
print(metrics)

{'accuracy': 0.75, 'precision': 0.7527, 'recall': 0.75, 'f1': 0.7474}


# Test and use the fine-tuned model

In [ ]:
model = SetFitModel.from_pretrained('/content/setfit_2epochs')

preds = model.predict([
    'I cooked a delicious meal today!',
])
print('Prediction:', np.array(preds)[0])

Prediction: 1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
preds = model.predict([
    'Some mornings are for sad songs and earl grey tea.',
])
print('Prediction:', np.array(preds)[0])

Prediction: 0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
